# Train Extra Trees

Reviewed by Muneeb Shakoor and the team on 22 October 2026. Prepared with AI assistance. See [AI use](../docs/AI_USE.md).

## Use the frozen settings

Extra Trees led the repeated search checks. Reproduce its exact settings here. This is one training reproduction, not another large search.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/review_models.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the project.')
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'mushrooms'), str(ROOT / 'citibike'), str(ROOT)]
import pandas as pd
from IPython.display import display
from search import build_model, read_partition, threshold_for_recall, metrics
from threadpoolctl import threadpool_limits
import joblib
frozen = json.loads((ROOT / 'mushrooms/frozen_candidates.json').read_text())
config = frozen['precalibration_selection']['candidate_configurations']['v3_extra_trees_0094']['config']
X, y = read_partition('search')
display(config['params'])

{'n_estimators': 600,
 'max_depth': None,
 'min_samples_leaf': 2,
 'max_features': 0.7007270656907509,
 'criterion': 'log_loss',
 'class_weight': None,
 'bootstrap': True,
 'max_samples': 0.9576950747515476}

## Fit the pipeline

Filling, encoding and the 600 trees learn from the 2,559 search rows. Save this reproduction separately. The app's original frozen model is unchanged.

In [2]:
model = build_model(config, X)
with threadpool_limits(limits=2):
    model.fit(X, y)
model.named_steps['model'].set_params(n_jobs=1)
OUT = ROOT / 'reports/reproduction/mushrooms'
OUT.mkdir(parents=True, exist_ok=True)
joblib.dump(model, OUT / 'extra_trees.joblib', compress=3)
print('Fitted and saved:', OUT / 'extra_trees.joblib')

C:\Users\tomis\CloudAI-research\team-review-2026-10-11\.venv-check\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Fitted and saved: C:\Users\tomis\CloudAI-research\team-handoff-2026-10-12\project\reports\reproduction\mushrooms\extra_trees.joblib


## Choose this fit's threshold

Use the separate 640 calibration rows. A new fit needs its own threshold. Catching all calibration poison labels does not guarantee catching all future poison labels.

In [3]:
cal_X, cal_y = read_partition('calibration')
with threadpool_limits(limits=1):
    scores = model.predict_proba(cal_X)[:, 1]
threshold, _, _ = threshold_for_recall(cal_y, scores, 1.0)
record = {'threshold': threshold, 'calibration': metrics(cal_y, scores, threshold),
          'final_test_scored': False, 'confirmation_rescored': False}
(OUT / 'reproduction.json').write_text(json.dumps(record, indent=2))
display(record)
print('Read the independent development trade-off in notebook 09.')

{'threshold': 0.0398015873015873,
 'calibration': {'ap': 0.8099185659449056,
  'roc_auc': 0.8529735454130154,
  'log_loss': 0.45530599411889405,
  'accuracy': 0.40625,
  'poison_recall': 1.0,
  'edible_specificity': 0.04522613065326633,
  'poison_precision': 0.3890675241157556,
  'threshold': 0.0398015873015873,
  'tn': 18,
  'fp': 380,
  'fn': 0,
  'tp': 242},
 'final_test_scored': False,
 'confirmation_rescored': False}

Read the independent development trade-off in notebook 09.
